<a href="https://colab.research.google.com/github/donjadao/lis4693/blob/main/Lab-3/Lab_Assignment_3_Data_Visualization_using_Altair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing panda, altair, and vega libraries

In [1]:
import pandas as pd
import altair as alt
from vega_datasets import data as vega_data

Loading seattle data

In [2]:
import requests
import io

url = "https://raw.githubusercontent.com/manika-lamba/SP26-LIS4_5693/refs/heads/main/lab-assignments/lab-3/Seattle-Library_2015-2021.csv"
response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors
text = response.text

save the data as a dataframe

In [3]:
seattle_df = pd.read_csv(io.StringIO(text))
seattle_df.head()

,Title,Creator,MaterialType,Checkouts,CheckoutYear,CheckoutMonth,Publisher,PublicationYear,Subjects,UsageClass,CheckoutType
0,Frog and toad all year / by Arnold Lobel.,"Lobel, Arnold",BOOK,34,2016,10,"Harper & Row,",c1976.,"Friendship Fiction, Frogs Juvenile fiction, To...",Physical,Horizon
1,"My brilliant friend : childhood, adolescence /...","Ferrante, Elena",BOOK,110,2016,10,"Europa Editions,",2012,"Friendship Fiction, Naples Italy Fiction",Physical,Horizon
2,Star trek [videorecording] / Paramount ; Spygl...,NaN,VIDEODISC,36,2016,10,"Paramount Home Entertainment,",c2009.,"Kirk James T 2233 2371 Drama, Spock Mr Drama, ...",Physical,Horizon
3,The Man in the High Castle,Philip K. Dick,EBOOK,63,2016,10,Houghton Mifflin Harcourt Trade and Reference,2012,"Fiction, Science Fiction",Digital,OverDrive
4,"The Fifth Season: Broken Earth Series, Book 1",N. K. Jemisin,EBOOK,44,2016,10,"Hachette Digital, Inc.",2015,"Fantasy, Fiction, Thriller",Digital,OverDrive


Who is the most popular author since every entry in this dataset is when i book is checked out. I am just going to see every count an author shows up.

In [5]:
author_counts = (
    seattle_df.groupby('Creator')
    .size()
    .reset_index(name='counts')
    .sort_values('counts', ascending=False)
)

author_counts

,Creator,counts
8861,"Willems, Mo",3010
1992,"Davis, Jim, 1945 July 28-",2381
7622,"Seuss, Dr.",2327
8046,"Stilton, Geronimo",1991
3582,"Holm, Jennifer L.",1683
...,...,...
13,ANDIE J. CHRISTOPHER,1
9109,"Zuckoff, Mitchell",1
9110,"Zuill, Andrea",1
9111,"Zuill, Andrea,",1


I am just doing a test to see it in graph form

In [6]:
author_popularity = author_counts.sort_values('counts', ascending=True)

alt.Chart(author_popularity.tail(25)).mark_bar().encode(
    x=alt.X('counts:Q', title='Total Checkouts'),
    y=alt.Y('Creator:N', sort='-x', title='Author'),
    tooltip=['Creator', 'counts']
).properties(
    title='Most Popular Authors by Total Checkouts'
)

alt.Chart(...)

I used to read a lot of geronimo stilton books as a kid so I am intersted if seattle likes the same titles I do, or if I remember the most checked out titles.

In [7]:
stilton_books = (
    seattle_df[seattle_df['Creator'] == 'Stilton, Geronimo']
    .groupby('Title')
    .size()
    .reset_index(name='counts')
    .sort_values('counts', ascending=False)
)

stilton_books

,Title,counts
109,The hunt for the curious cheese : plus a bonus...,48
110,The hunt for the golden book / Geronimo Stilto...,42
111,The hunt for the secret papyrus : plus a bonus...,41
87,The amazing voyage : the third adventure in th...,40
123,The race against time : the third journey thro...,38
...,...,...
105,The haunted dinosaur / Geronimo Stilton ; [ill...,1
126,The secret of Cacklefur Castle / Geronimo Stil...,1
143,Thea Stilton and the dancing shadows / Geronim...,1
148,Watch your tail! / text by Geronimo Stilton ; ...,1


Next I am trying a circle plot of the books popularity:

I am not sure what to do for size or opacity, but

In [8]:
stilton_circle = (
    alt.Chart(stilton_books)
    .mark_circle(size=200, opacity=1, color='orange')
    .encode(
        x=alt.X('counts:Q', title='Total Checkouts'),
        y=alt.Y('Title:N', sort='-x', title='Book Title'),
        tooltip=['Title', 'counts']
    )
    .properties(title='Popularity of Geronimo Stilton Books')
    .interactive()
)

stilton_circle

alt.Chart(...)

It is a big graph. So I will limit it to the top 25 books. Additionally, I want to see if certain books get checked out in other years more than others. A line graph going across time will look better for this kind of data.

In [9]:
stilton_by_year = (
    seattle_df[seattle_df['Creator'] == 'Stilton, Geronimo']
    .groupby(['Title', 'CheckoutYear'])
    .size()
    .reset_index(name='counts')
)
stilton_top25_titles = (
    stilton_by_year.groupby('Title')['counts']
    .sum()
    .sort_values(ascending=False)
    .head(25)
    .index
)

stilton_top25 = stilton_by_year[stilton_by_year['Title'].isin(stilton_top25_titles)]

In [11]:
stilton_top25_chart = (
    alt.Chart(stilton_top25)
    .mark_line(point=True)
    .encode(
        x=alt.X('CheckoutYear:O', title='Year'),
        y=alt.Y('counts:Q', title='Total Checkouts'),
        color=alt.Color('Title:N', title='Book Title'),
        tooltip=['Title', 'CheckoutYear', 'counts']
    )
    .properties(
        title='Top 25 Geronimo Stilton Books: Checkouts by Year',
    )
    .interactive()
)

stilton_top25_chart

alt.Chart(...)

This is still super unreadable, so I learned how to add onto properties:

In [12]:
stilton_top25_chart = (
    alt.Chart(stilton_top25)
    .mark_line(point=True)
    .encode(
        x=alt.X('CheckoutYear:O', title='Year'),
        y=alt.Y('counts:Q', title='Total Checkouts'),
        color=alt.Color('Title:N', title='Book Title'),
        tooltip=['Title', 'CheckoutYear', 'counts']
    )
    .properties(
        title='Top 25 Geronimo Stilton Books — Checkouts by Year',
        width=400,
        height=500
    )
    .interactive()
)

stilton_top25_chart

alt.Chart(...)

You can hover over the small dots but they are still pretty small.

I have a better understanding of what to do now, Next I will get a graph of what months are the most popular to check out books. Additionally, I want to create a stacked bar graph of this

In [15]:
month_year_counts = (
    seattle_df.groupby(['CheckoutMonth', 'CheckoutYear'])
    .size()
    .reset_index(name='counts')
)

In [17]:
month_year_counts = seattle_df.groupby(['CheckoutMonth', 'CheckoutYear']).size().reset_index(name='counts')

selection = alt.selection_point(fields=['CheckoutYear'], bind='legend')

alt.Chart(month_year_counts).mark_bar().encode(
    y=alt.Y('CheckoutMonth:O', sort=list(range(1,13))),
    x='counts:Q',
    color=alt.Color('CheckoutYear:N',
                    sort=alt.EncodingSortField('CheckoutYear', order='descending')),
    tooltip=['CheckoutMonth', 'counts', 'CheckoutYear'],
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    order='CheckoutYear'
).add_params(
    selection
)

alt.Chart(...)

It's weird reading months as numbers so I will fix the label to be the actual months

In [19]:
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

month_year_counts['MonthName'] = month_year_counts['CheckoutMonth'].map(month_map)

In [20]:
selection = alt.selection_point(fields=['CheckoutYear'], bind='legend')

alt.Chart(month_year_counts).mark_bar().encode(
    y=alt.Y('MonthName:N',
            sort=['January','February','March','April','May','June',
                  'July','August','September','October','November','December']),
    x='counts:Q',
    color=alt.Color('CheckoutYear:N',
                    sort=alt.EncodingSortField('CheckoutYear', order='descending')),
    tooltip=['MonthName', 'counts', 'CheckoutYear'],
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    order='CheckoutYear'
).add_params(
    selection
)

alt.Chart(...)

I am going to guess that january is the most popular date to check out books since its the new year and people might make goals to read more. Its also winter and everyone is indoors.

Finally horizontal concatinations:

In [21]:
(month_year_horizontal | stilton_top25_chart)

alt.HConcatChart(...)

# Reflection

What went well?
What did not go well or what challenges you encountered?

There's a lot of playing around that goes in to this and learning as you go. You don't get what you want in a single go. I'm also not sure, if I should be sizing with properties. It also did a lot of scrolling, but eventually I just did a google search of how to add a title to a graph in altair, but also I learned how to do the sizing. It annoyed me a lot not knowing how to add a title. I think the more challenging part is just finding out what information might be usable or intersting to look at. Some of the interactivity is a little troublesome to look at, the scrolling/zooming can be a little annoying using it in google collab, but in a regular website, it might be easier to use.